In [ ]:
import pandas as pd

# 1. Load the CSV file
# Replace 'your_file.csv' with the actual path to your file
file_path = 'dataset.csv'
df = pd.read_csv(file_path)

# 2. Check the number of rows
# df.shape returns (rows, columns)
num_rows = df.shape[0]
print(f"Total Number of Rows: {num_rows}")

# 3. Check the headers
print("\nHeaders:")
print(df.columns.tolist())

# 4. Show the first 20 rows
print("\nFirst 20 Rows:")
print(df.head(20))

Total Number of Rows: 100000

Headers:
['CommentText', 'Sentiment', 'Likes', 'Replies', 'CountryCode', 'CategoryID']

First 20 Rows:
                                          CommentText Sentiment  Likes  \
0                     Anyone know what movie this is?   Neutral      0   
1   The fact they're holding each other back while...  Positive      0   
2                         waiting next video will be?   Neutral      1   
3                                    Dei løk de seim😂   Neutral      0   
4        Number two because it looks the best with it  Positive      0   
5   Thank God we don’t have to listen to his drive...  Positive      0   
6   Very similar thing happened to me! We lived ne...  Positive      0   
7                                    im about to cry😢  Negative      0   
8                                           Big deal.   Neutral      0   
9                                 Respect for physics  Positive      0   
10  now my book is crystal clear.I can understand ...

In [ ]:
import pandas as pd
from transformers import pipeline
from tqdm import tqdm
import torch
import os

# -----------------------------
# 1. Config
# -----------------------------
MODEL_NAME = "valhalla/distilbart-mnli-12-3"
BATCH_SIZE = 64
CHUNK_SIZE = 20000
INPUT_FILE = "dataset.csv"
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

candidate_labels = ["Question", "Appreciation", "Spam", "Complaint"]

# -----------------------------
# 2. Device setup
# -----------------------------
device = 0 if torch.cuda.is_available() else -1
print("Using GPU" if device == 0 else "Using CPU")

# -----------------------------
# 3. Load model
# -----------------------------
classifier = pipeline(
    "zero-shot-classification",
    model=MODEL_NAME,
    device=device
)

# -----------------------------
# 4. Load dataset
# -----------------------------
df = pd.read_csv(INPUT_FILE)
texts = df['CommentText'].fillna("").astype(str).tolist()

total_rows = len(texts)
print(f"Total comments: {total_rows}")

# -----------------------------
# 5. Process in chunks
# -----------------------------
for chunk_start in range(0, total_rows, CHUNK_SIZE):
    chunk_end = min(chunk_start + CHUNK_SIZE, total_rows)

    print(f"\nProcessing rows {chunk_start} → {chunk_end}")

    chunk_texts = texts[chunk_start:chunk_end]
    chunk_df = df.iloc[chunk_start:chunk_end].copy()

    results = []

    # Batch loop
    for i in tqdm(range(0, len(chunk_texts), BATCH_SIZE),
                  desc=f"Chunk {chunk_start//CHUNK_SIZE + 1}"):

        batch = chunk_texts[i:i + BATCH_SIZE]

        outputs = classifier(
            batch,
            candidate_labels,
            truncation=True
        )

        for out in outputs:
            results.append({
                "intent_label": out["labels"][0],
                "intent_confidence": round(out["scores"][0], 4)
            })

    # Merge results
    chunk_results = pd.concat([chunk_df, pd.DataFrame(results)], axis=1)
    # theres some issue here in merging , check it out

    # Save checkpoint
    output_file = f"{OUTPUT_DIR}/chunk_{chunk_start}_{chunk_end}.csv"
    chunk_results.to_csv(output_file, index=False)

    print(f"✅ Saved: {output_file}")

print("\n🎉 ALL CHUNKS COMPLETED!")

Using GPU


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/283 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


...

✅ Saved: outputs/chunk_40000_60000.csv

Processing rows 60000 → 80000


Chunk 4: 100%|██████████| 313/313 [18:22<00:00,  3.52s/it]


✅ Saved: outputs/chunk_60000_80000.csv

Processing rows 80000 → 100000


Chunk 5: 100%|██████████| 313/313 [18:17<00:00,  3.51s/it]

✅ Saved: outputs/chunk_80000_100000.csv

🎉 ALL CHUNKS COMPLETED!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import glob
import os

# -----------------------------
# 1. Path to chunk files
# -----------------------------
INPUT_DIR = "outputs"   # folder where chunks are saved
OUTPUT_FILE = "final_sorted_results.csv"

# -----------------------------
# 2. Load all CSV files
# -----------------------------
files = glob.glob(os.path.join(INPUT_DIR, "*.csv"))

print(f"Found {len(files)} chunk files")

df_list = []
for file in files:
    print(f"Loading: {file}")
    df_list.append(pd.read_csv(file))

# -----------------------------
# 3. Combine
# -----------------------------
df = pd.concat(df_list, ignore_index=True)

print(f"Total rows after merge: {len(df)}")

# -----------------------------
# 4. Sort by confidence
# -----------------------------
df_sorted = df.sort_values(
    by="intent_confidence",
    ascending=False
)

# -----------------------------
# 5. Save final file
# -----------------------------
df_sorted.to_csv(OUTPUT_FILE, index=False)

print(f"\n✅ Final sorted file saved as: {OUTPUT_FILE}")

Found 5 chunk files
Loading: outputs/fixed_chunk2.csv
Loading: outputs/chunk_0_20000.csv
Loading: outputs/fixed_chunk3.csv
Loading: outputs/fixed_chunk4.csv
Loading: outputs/fixed_chunk5.csv
Total rows after merge: 100000

✅ Final sorted file saved as: final_sorted_results.csv


In [ ]:
import pandas as pd

# Load broken file
df = pd.read_csv("/content/outputs/chunk_80000_100000.csv")

# -----------------------------
# 1. Split original data
# -----------------------------
original = df[df["CommentText"].notna()].copy()

# -----------------------------
# 2. Extract predictions
# -----------------------------
preds = df[df["intent_label"].notna()][
    ["intent_label", "intent_confidence"]
].copy()

# -----------------------------
# 3. Reset index for alignment
# -----------------------------
original = original.reset_index(drop=True)
preds = preds.reset_index(drop=True)

# -----------------------------
# 4. Safety check
# -----------------------------
print("Original:", len(original))
print("Predictions:", len(preds))

assert len(original) == len(preds), "❌ Mismatch — do NOT proceed!"

# -----------------------------
# 5. Assign instead of concat (cleaner)
# -----------------------------
original["intent_label"] = preds["intent_label"]
original["intent_confidence"] = preds["intent_confidence"]

# -----------------------------
# 6. Save fixed file
# -----------------------------
original.to_csv("/content/outputs/fixed_chunk5.csv", index=False)

print("✅ Fixed file saved as fixed_chunk2.csv")

Original: 20000
Predictions: 20000
✅ Fixed file saved as fixed_chunk2.csv
